# 2Baseline Model: Historical, Parametric, and Quantile Benchmarks

This notebook produces strictly out-of-sample benchmark forecasts for the next-month stock return distribution. It is a **model-production notebook**, not an asset-pricing notebook: no portfolio return is used to choose a model here.

The notebook contains five benchmark families:

1. historical empirical VaR/ES;
2. rolling Gaussian;
3. rolling Student-$t$ using rolling moments;
4. conditional one-component Normal and Student-$t$ neural networks using the same predictors as the MDN;
5. joint linear quantile regression and an optional quantile MLP.

All models return the same economic units: raw decimal monthly returns, where `0.05` means 5%. To match `3ES_Calculation.ipynb`, `var_5` is the raw 5% return quantile while `es_5` is the positive loss magnitude $-E[R\mid R\leq VaR_{5\%}]$.


## 1. Assumed Input Contract

The cleaned input is assumed to contain one unique row per `(permno, mthcaldt)` and the following fields:

- `permno`: stock identifier;
- `mthcaldt`: forecast formation month $t$;
- `mthret`: raw return observed in month $t$, available when the forecast is formed;
- `target_ret_final`: raw return in month $t+1$;
- cleaned numeric micro and macro predictors observed by month $t$.

The real run reads `micro_data.parquet` and `MacroFactor_McCracken.parquet` separately, merges them by formation month, and uses the explicit feature contract in the configuration cell. `target_ret_final` is asserted to be label-only.


In [1]:
from pathlib import Path
import importlib.util
import copy
import json
import math
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, t as student_t

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cpu


## 2. Configuration

`RUN_MODE='smoke'` uses synthetic data, `quick` uses a bounded stock sample and 24 OOS months, and `full` runs the complete production sample. The environment and paths are detected automatically for local, Colab, and Kaggle runs.


In [2]:
# ---------- Run mode ----------
RUN_MODE = 'quick'  # 'smoke', 'quick', or 'full'
USE_SYNTHETIC_DATA = (RUN_MODE == 'smoke')
SAVE_SYNTHETIC_OUTPUT = False

# ---------- Platform and paths ----------
ENV_OVERRIDE = None  # None, 'local', 'colab', or 'kaggle'
KAGGLE_DATASET_SLUG = 'mdndata-1990'
COLAB_DATA_DIR = Path('/content/drive/MyDrive/PricingTailRisk_MDN/data')
COLAB_OUTPUT_DIR = Path('/content/drive/MyDrive/PricingTailRisk_MDN/output/baseline')

def detect_environment() -> str:
    if ENV_OVERRIDE is not None:
        return ENV_OVERRIDE.lower()
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/input').exists():
        return 'kaggle'
    if importlib.util.find_spec('google.colab') is not None:
        return 'colab'
    return 'local'

def find_local_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'micro_data.parquet').exists():
            return candidate.resolve()
    raise FileNotFoundError('Cannot locate the project root containing data/micro_data.parquet.')

def find_kaggle_input(filename: str) -> Path:
    preferred = Path('/kaggle/input') / KAGGLE_DATASET_SLUG / filename
    if preferred.exists():
        return preferred
    matches = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(f'{filename} was not found under /kaggle/input.')
    raise RuntimeError(f'Multiple Kaggle inputs named {filename}: {matches}')

ENV = detect_environment()
if ENV == 'local':
    PROJECT_ROOT = find_local_project_root()
    MICRO_DATA_PATH = PROJECT_ROOT / 'data' / 'micro_data.parquet'
    MACRO_DATA_PATH = PROJECT_ROOT / 'data' / 'MacroFactor_McCracken.parquet'
    OUTPUT_DIR = PROJECT_ROOT / 'output' / 'baseline'
elif ENV == 'kaggle':
    PROJECT_ROOT = Path('/kaggle/working')
    MICRO_DATA_PATH = find_kaggle_input('micro_data.parquet')
    MACRO_DATA_PATH = find_kaggle_input('MacroFactor_McCracken.parquet')
    OUTPUT_DIR = Path('/kaggle/working/baseline')
elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content')
    MICRO_DATA_PATH = COLAB_DATA_DIR / 'micro_data.parquet'
    MACRO_DATA_PATH = COLAB_DATA_DIR / 'MacroFactor_McCracken.parquet'
    OUTPUT_DIR = COLAB_OUTPUT_DIR
else:
    raise ValueError(f'Unsupported environment: {ENV}')

OUTPUT_PATH = OUTPUT_DIR / f'baseline_oos_forecasts_{RUN_MODE}.parquet'
CONFIG_PATH = OUTPUT_DIR / f'baseline_config_{RUN_MODE}.json'

# ---------- Required columns ----------
ID_COL = 'permno'
DATE_COL = 'mthcaldt'
CURRENT_RETURN_COL = 'mthret'
TARGET_COL = 'target_ret_final'

# Explicit feature contract shared with 3MDN_model.ipynb.
MICRO_FEATURES = [
    'mthret', 'bm', 'evm', 'pe_exi', 'ps', 'pcf', 'pfcf', 'npm',
    'opmad', 'gpm', 'cfm', 'roe', 'aftret_invcapx', 'gprof',
    'debt_at', 'debt_ebitda', 'short_debt', 'ocf_lct', 'de_ratio',
    'cash_ratio', 'curr_ratio', 'at_turn', 'rect_turn', 'pay_turn',
    'sale_equity', 'rd_sale', 'adv_sale', 'staff_sale', 'accrual',
    'MOM12m', 'mu_bench', 'sigma_bench', 'mthcap_log',
]
RAW_SCALE_MICRO_FEATURES = [
    'mthret', 'MOM12m', 'mu_bench', 'sigma_bench', 'mthcap_log',
]
RANK_MICRO_FEATURES = [
    col for col in MICRO_FEATURES if col not in RAW_SCALE_MICRO_FEATURES
]
MACRO_FEATURES = [
    'PAYEMS', 'UNRATE', 'CLAIMSx', 'INDPRO', 'DPCERA3M086SBEA',
    'AMDMNOx', 'ISRATIOx', 'HOUST', 'M2SL', 'NONBORRES', 'BUSLOANS',
    'FEDFUNDS', 'CP3Mx', 'UEMPMEAN', 'CPIAUCSL', 'VIXCLSx', 'BAAFFM',
    'TERM_SPREAD', 'CREDIT_SPREAD',
]

# ---------- Forecast objects ----------
ALPHAS = (0.01, 0.05, 0.10)
ALPHA_SUFFIX = {0.01: '1', 0.05: '5', 0.10: '10'}
QUANTILE_LEVELS = np.array([
    0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50,
    0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.9975, 0.999
], dtype=float)

# ---------- Time design ----------
INITIAL_TRAIN_MONTHS = 180
REAL_OOS_START = None  # inferred after initial train + validation months
REAL_OOS_END = None  # e.g. pd.Timestamp('2025-12-31')
REFIT_MONTHS = 12
VALIDATION_MONTHS = 12
MIN_NEURAL_TRAIN_MONTHS = 36
QUICK_MAX_STOCKS = 250
QUICK_OOS_MONTHS = 24

# ---------- Historical baselines ----------
HISTORY_WINDOW_MONTHS = 120
MIN_HISTORY_MONTHS = 36
T_DF_MIN = 4.1
T_DF_MAX = 100.0

# ---------- Neural baselines ----------
STANDARDIZE_FEATURES_WITH_TRAINING_DATA = True
SIGMA_FLOOR = 0.005  # raw-return units: 0.5% monthly volatility
NU_MIN = 3.0
NU_MAX = 30.0
HIDDEN_DIMS = (64, 32)
DROPOUT = 0.10
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
EARLY_STOPPING_PATIENCE = 8
SEED = 42

RUN_HISTORICAL_BASELINES = True
RUN_CONDITIONAL_NORMAL_K1 = True
RUN_CONDITIONAL_STUDENT_K1 = True
RUN_LINEAR_QUANTILE = True
RUN_QUANTILE_MLP = False

if RUN_MODE == 'smoke':
    MAX_EPOCHS = 4
    BATCH_SIZE = 1024
    VALIDATION_MONTHS = 12
elif RUN_MODE == 'quick':
    MAX_EPOCHS = 10
else:
    MAX_EPOCHS = 100

print({
    'run_mode': RUN_MODE,
    'synthetic': USE_SYNTHETIC_DATA,
    'environment': ENV,
    'micro_data_path': str(MICRO_DATA_PATH),
    'macro_data_path': str(MACRO_DATA_PATH),
    'output_path': str(OUTPUT_PATH),
})


{'run_mode': 'quick', 'synthetic': False, 'environment': 'local', 'micro_data_path': '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet', 'macro_data_path': '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor_McCracken.parquet', 'output_path': '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output/baseline/baseline_oos_forecasts_quick.parquet'}


## 3. Load or Generate the Model Panel


In [3]:
def make_synthetic_panel(n_stocks: int = 60, n_months: int = 84, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.date_range('2010-01-31', periods=n_months, freq='ME')
    stock_ids = np.arange(10000, 10000 + n_stocks)

    macro = pd.DataFrame({
        DATE_COL: dates,
        'macro_term': np.sin(np.arange(n_months) / 10) + rng.normal(0, 0.15, n_months),
        'macro_default': rng.normal(0, 1, n_months),
        'macro_market_vol': np.clip(rng.lognormal(-2.2, 0.35, n_months), 0.03, 0.40),
    })

    rows = []
    firm_size = rng.normal(10, 1.5, n_stocks)
    firm_bm = rng.normal(0, 1, n_stocks)
    firm_quality = rng.normal(0, 1, n_stocks)

    for j, permno in enumerate(stock_ids):
        previous_return = 0.0
        for t_idx, date in enumerate(dates):
            macro_row = macro.iloc[t_idx]
            conditional_mean = (
                0.006 + 0.002 * firm_bm[j] + 0.003 * firm_quality[j]
                - 0.02 * macro_row['macro_market_vol']
            )
            conditional_vol = np.clip(
                0.06 + 0.035 * abs(firm_quality[j]) + 0.55 * macro_row['macro_market_vol'],
                0.04, 0.35
            )
            ret = conditional_mean + conditional_vol * rng.standard_t(df=6) / math.sqrt(6 / 4)
            ret = float(np.clip(ret, -0.95, 2.0))
            rows.append({
                ID_COL: int(permno),
                DATE_COL: date,
                CURRENT_RETURN_COL: ret,
                'micro_size': firm_size[j] + rng.normal(0, 0.05),
                'micro_bm': firm_bm[j] + rng.normal(0, 0.10),
                'micro_quality': firm_quality[j] + rng.normal(0, 0.10),
                'micro_momentum': previous_return,
                'macro_term': macro_row['macro_term'],
                'macro_default': macro_row['macro_default'],
                'macro_market_vol': macro_row['macro_market_vol'],
            })
            previous_return = ret

    result = pd.DataFrame(rows).sort_values([ID_COL, DATE_COL]).reset_index(drop=True)
    result[TARGET_COL] = result.groupby(ID_COL)[CURRENT_RETURN_COL].shift(-1)
    return result

if USE_SYNTHETIC_DATA:
    panel_df = make_synthetic_panel(seed=SEED)
    MICRO_FEATURES = ['micro_size', 'micro_bm', 'micro_quality', 'micro_momentum']
    RAW_SCALE_MICRO_FEATURES = MICRO_FEATURES.copy()
    RANK_MICRO_FEATURES = []
    MACRO_FEATURES = ['macro_term', 'macro_default', 'macro_market_vol']
else:
    for data_path in (MICRO_DATA_PATH, MACRO_DATA_PATH):
        if not data_path.exists():
            raise FileNotFoundError(
                f'Missing {data_path}. On Colab, put both parquet files in {COLAB_DATA_DIR}; '
                'on Kaggle, attach the dataset containing both files.'
            )

    micro_required = [
        ID_COL, DATE_COL, CURRENT_RETURN_COL, TARGET_COL, 'ffi49',
    ] + [col for col in MICRO_FEATURES if col != CURRENT_RETURN_COL]
    macro_required = [DATE_COL] + MACRO_FEATURES
    micro_df = pd.read_parquet(MICRO_DATA_PATH, columns=micro_required)
    macro_df = pd.read_parquet(MACRO_DATA_PATH, columns=macro_required)

    micro_df[DATE_COL] = pd.to_datetime(micro_df[DATE_COL]) + pd.offsets.MonthEnd(0)
    macro_df[DATE_COL] = pd.to_datetime(macro_df[DATE_COL]) + pd.offsets.MonthEnd(0)
    if micro_df.duplicated([ID_COL, DATE_COL]).any():
        raise ValueError('micro_data contains duplicate (permno, mthcaldt) rows.')
    if macro_df[DATE_COL].duplicated().any():
        raise ValueError('MacroFactor_McCracken contains duplicate months.')

    # Monthly cross-sectional ranks use only information available in formation month t.
    ranked = micro_df.groupby(DATE_COL)[RANK_MICRO_FEATURES].rank(pct=True) * 2.0 - 1.0
    micro_df[RANK_MICRO_FEATURES] = ranked.fillna(0.0).to_numpy()
    panel_df = micro_df.merge(
        macro_df, on=DATE_COL, how='left', validate='many_to_one'
    )

FEATURE_COLS = MICRO_FEATURES + MACRO_FEATURES
if TARGET_COL in FEATURE_COLS:
    raise AssertionError(f'{TARGET_COL} must be a label only, never a feature.')
if len(FEATURE_COLS) != len(set(FEATURE_COLS)):
    raise AssertionError('Duplicate feature names found.')

panel_df = panel_df.sort_values([DATE_COL, ID_COL]).reset_index(drop=True)
if RUN_MODE == 'quick' and not USE_SYNTHETIC_DATA:
    rng = np.random.default_rng(SEED)
    stock_ids = np.sort(panel_df[ID_COL].dropna().unique())
    selected_ids = rng.choice(
        stock_ids, size=min(QUICK_MAX_STOCKS, len(stock_ids)), replace=False
    )
    panel_df = panel_df.loc[panel_df[ID_COL].isin(selected_ids)].copy()

available_dates = pd.DatetimeIndex(sorted(pd.to_datetime(panel_df[DATE_COL]).unique()))
if USE_SYNTHETIC_DATA:
    OOS_START = pd.Timestamp('2015-01-31')
else:
    inferred_position = INITIAL_TRAIN_MONTHS + VALIDATION_MONTHS
    if inferred_position >= len(available_dates):
        raise ValueError('Not enough months for the requested train/validation design.')
    OOS_START = REAL_OOS_START or pd.Timestamp(available_dates[inferred_position])
OOS_END = REAL_OOS_END or pd.Timestamp(available_dates[-1])
if RUN_MODE == 'quick':
    quick_dates = available_dates[available_dates >= OOS_START][:QUICK_OOS_MONTHS]
    if len(quick_dates):
        OOS_END = min(pd.Timestamp(OOS_END), pd.Timestamp(quick_dates[-1]))
print('Panel shape:', panel_df.shape)
print('Date range:', panel_df[DATE_COL].min(), 'to', panel_df[DATE_COL].max())
print('Feature count:', len(FEATURE_COLS))
display(panel_df.head())


Panel shape: (24838, 56)
Date range: 1991-01-31 00:00:00 to 2024-12-31 00:00:00
Feature count: 52


,permno,mthcaldt,mthret,target_ret_final,ffi49,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,MOM12m,mu_bench,sigma_bench,mthcap_log,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
70,10265,1991-01-31,0.025000,0.146341,0,0.005702,-0.009186,0.000950,-0.004118,0.002217,0.032309,0.000317,-0.000950,-0.004118,-0.029458,-0.000634,0.008236,0.001584,0.002217,0.000317,-0.025974,0.035793,0.000317,-0.020272,0.038961,0.002534,0.004118,0.012987,0.002534,-0.292683,-0.281596,-0.168514,-0.002217,-0.284673,-0.021474,0.089477,10.331309,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
193,10725,1991-01-31,0.056604,0.207558,0,0.681977,-0.361419,0.000950,-0.481787,0.002217,0.032309,0.496991,0.644599,-0.257523,-0.029458,0.197339,-0.789357,-0.908774,-0.633830,0.609756,-0.729490,0.035793,0.855559,-0.020272,0.038961,-0.875831,0.004118,0.012987,-0.678809,-0.292683,-0.281596,0.678175,-0.002217,-0.077467,0.005295,0.131134,11.481662,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
362,11353,1991-01-31,-0.093790,0.041322,34,-0.351283,-0.443142,0.370922,0.780171,0.605955,0.491289,0.901806,0.922711,0.720621,0.763700,0.277795,0.995565,0.987963,0.854926,-0.542604,-0.968324,0.681977,-0.986696,-0.668039,-0.884701,0.888502,0.790941,-0.581248,0.373456,0.450111,0.501425,-0.168514,-0.524232,0.917278,0.054227,0.127912,13.474933,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
403,11513,1991-01-31,-0.061225,0.108696,0,0.005702,-0.009186,0.000950,-0.004118,0.002217,0.032309,0.000317,-0.000950,-0.004118,-0.029458,-0.000634,0.008236,0.001584,0.002217,0.000317,-0.025974,0.035793,0.000317,-0.020272,0.038961,0.002534,0.004118,0.012987,0.002534,-0.292683,-0.281596,-0.168514,-0.002217,-0.183333,-0.019407,0.071745,10.105285,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
612,13354,1991-01-31,0.255102,0.130081,14,-0.267659,-0.220779,0.321508,-0.866329,0.275261,-0.428571,-0.705417,0.234083,-0.422236,-0.409566,0.355084,0.372189,-0.372189,0.972125,0.837821,-0.910041,-0.692113,-0.992398,-0.370922,-0.194805,-0.381058,0.312005,0.730757,0.270827,0.614191,-0.281596,-0.168514,0.473551,-0.453770,-0.019821,0.157909,12.968049,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35


## 4. Validate Timing and Schema

The checks below are intentionally strict. If the target is not exactly the next observation of the raw stock return, stop and repair the data notebook before training a benchmark.


In [4]:
panel_df = panel_df.copy()
panel_df[DATE_COL] = pd.to_datetime(panel_df[DATE_COL]) + pd.offsets.MonthEnd(0)
panel_df = panel_df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)
panel_df['target_date'] = panel_df[DATE_COL] + pd.offsets.MonthEnd(1)

required_cols = [ID_COL, DATE_COL, CURRENT_RETURN_COL, TARGET_COL] + FEATURE_COLS
missing_cols = [col for col in required_cols if col not in panel_df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

if panel_df.duplicated([ID_COL, DATE_COL]).any():
    duplicates = panel_df.loc[panel_df.duplicated([ID_COL, DATE_COL], keep=False), [ID_COL, DATE_COL]]
    raise ValueError(f'Duplicate stock-month rows found. Example: {duplicates.head()}')

expected_target = panel_df.groupby(ID_COL)[CURRENT_RETURN_COL].shift(-1)
next_observed_date = panel_df.groupby(ID_COL)[DATE_COL].shift(-1)
# The $5 formation-price filter can remove an intermediate row. Validate the
# target against the next row only when that row is the next calendar month.
alignment_mask = (
    panel_df[TARGET_COL].notna()
    & next_observed_date.eq(panel_df['target_date'])
)
max_alignment_error = (
    expected_target.loc[alignment_mask] - panel_df.loc[alignment_mask, TARGET_COL]
).abs().max()
if pd.notna(max_alignment_error) and max_alignment_error > 1e-10:
    raise ValueError(f'{TARGET_COL} is not the next-month {CURRENT_RETURN_COL}; max error={max_alignment_error}')

numeric_block = panel_df[FEATURE_COLS + [CURRENT_RETURN_COL, TARGET_COL]].replace([np.inf, -np.inf], np.nan)
feature_missing = numeric_block[FEATURE_COLS].isna().sum()
if feature_missing.sum() > 0:
    raise ValueError(f'Features still contain missing values: {feature_missing[feature_missing > 0]}')
return_missing = numeric_block.loc[panel_df[TARGET_COL].notna(), [CURRENT_RETURN_COL, TARGET_COL]].isna().sum()
if return_missing.sum() > 0:
    raise ValueError(f'Observed current/target returns contain non-finite values: {return_missing}')

panel_df = panel_df.loc[panel_df[TARGET_COL].notna()].copy()
panel_df = panel_df.loc[panel_df[DATE_COL].between(panel_df[DATE_COL].min(), OOS_END)].copy()

print('Schema and target alignment checks passed.')
print('OOS period:', OOS_START, 'to', OOS_END)
print('OOS rows:', int((panel_df[DATE_COL] >= OOS_START).sum()))


Schema and target alignment checks passed.
OOS period: 2007-01-31 00:00:00 to 2008-12-31 00:00:00
OOS rows: 1415


## 5. Distribution and Output Helpers


In [5]:
def safe_scale(value: float, floor: float = 1e-6) -> float:
    return float(max(float(value), floor))

def normal_var_and_loss_es(alpha: float, loc: float, scale: float) -> tuple[float, float]:
    scale = safe_scale(scale)
    z = norm.ppf(alpha)
    var_value = loc + scale * z
    tail_mean = loc - scale * norm.pdf(z) / alpha
    return float(var_value), float(-tail_mean)

def student_var_and_loss_es(
    alpha: float, loc: float, scale: float, degrees_of_freedom: float
) -> tuple[float, float]:
    scale = safe_scale(scale)
    degrees_of_freedom = float(max(degrees_of_freedom, 1.01))
    z = student_t.ppf(alpha, df=degrees_of_freedom)
    density = student_t.pdf(z, df=degrees_of_freedom)
    standard_tail_mean = -(
        (degrees_of_freedom + z ** 2) / (degrees_of_freedom - 1)
    ) * density / alpha
    var_value = loc + scale * z
    tail_mean = loc + scale * standard_tail_mean
    return float(var_value), float(-tail_mean)

def rolling_t_parameters(history: np.ndarray) -> tuple[float, float, float, float]:
    history = np.asarray(history, dtype=float)
    loc = float(np.mean(history))
    sample_std = safe_scale(np.std(history, ddof=1))
    centered = history - loc
    second_moment = float(np.mean(centered ** 2))
    fourth_moment = float(np.mean(centered ** 4))
    excess_kurtosis = fourth_moment / max(second_moment ** 2, 1e-12) - 3.0
    if excess_kurtosis > 0:
        degrees_of_freedom = 6.0 / excess_kurtosis + 4.0
    else:
        degrees_of_freedom = T_DF_MAX
    degrees_of_freedom = float(np.clip(degrees_of_freedom, T_DF_MIN, T_DF_MAX))
    scale = sample_std * math.sqrt((degrees_of_freedom - 2.0) / degrees_of_freedom)
    return loc, safe_scale(scale), degrees_of_freedom, sample_std

def base_forecast_row(row: pd.Series, model_id: str) -> dict:
    return {
        ID_COL: row[ID_COL],
        'forecast_date': row[DATE_COL],
        'target_date': row['target_date'],
        'realized_ret': float(row[TARGET_COL]),
        'model_id': model_id,
    }

def add_tail_statistics(output_row: dict, tail_values: dict[float, tuple[float, float]]) -> dict:
    for alpha, (var_value, es_value) in tail_values.items():
        suffix = ALPHA_SUFFIX[alpha]
        output_row[f'var_{suffix}'] = float(var_value)
        output_row[f'es_{suffix}'] = float(es_value)
    return output_row

def empty_forecast_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=[
        ID_COL, 'forecast_date', 'target_date', 'realized_ret', 'model_id',
        'pred_mean', 'pred_vol', 'var_1', 'es_1',
        'var_5', 'es_5', 'var_10', 'es_10'
    ])


## 6. Historical Empirical, Rolling Normal, and Rolling Student-$t$

All rolling windows end in month $t$ and therefore use only returns known when predicting month $t+1$. The Student-$t$ benchmark uses a scalable method-of-moments estimate of the degrees of freedom instead of fitting a nonlinear MLE for every stock-month.


In [6]:
def empirical_var_and_loss_es(history: np.ndarray, alpha: float) -> tuple[float, float]:
    ordered = np.sort(np.asarray(history, dtype=float))
    var_value = float(np.quantile(ordered, alpha, method='linear'))
    tail_mass = alpha * len(ordered)
    full_observations = int(math.floor(tail_mass))
    fractional_mass = tail_mass - full_observations
    tail_sum = float(ordered[:full_observations].sum())
    if full_observations < len(ordered):
        tail_sum += fractional_mass * float(ordered[full_observations])
    tail_mean = tail_sum / tail_mass
    return var_value, -tail_mean

def run_rolling_distribution_baselines(panel: pd.DataFrame) -> pd.DataFrame:
    output_rows = []

    grouped = panel.groupby(ID_COL, sort=False)
    for _, stock_df in tqdm(grouped, total=panel[ID_COL].nunique(), desc='Rolling baselines'):
        stock_df = stock_df.sort_values(DATE_COL).reset_index(drop=True)
        stock_returns = stock_df[CURRENT_RETURN_COL].to_numpy(dtype=float)
        stock_dates = pd.to_datetime(stock_df[DATE_COL]).to_numpy()

        for position, row in stock_df.iterrows():
            forecast_date = row[DATE_COL]
            if forecast_date < OOS_START or forecast_date > OOS_END:
                continue

            history_start_date = forecast_date - pd.offsets.MonthEnd(HISTORY_WINDOW_MONTHS - 1)
            history_mask = (stock_dates >= np.datetime64(history_start_date)) & (
                stock_dates <= np.datetime64(forecast_date)
            )
            history = stock_returns[history_mask]
            history = history[np.isfinite(history)]
            if len(history) < MIN_HISTORY_MONTHS:
                continue

            historical_mean = float(np.mean(history))
            historical_std = safe_scale(np.std(history, ddof=1))

            # Historical empirical distribution
            empirical_tail_values = {
                alpha: empirical_var_and_loss_es(history, alpha) for alpha in ALPHAS
            }
            empirical_row = base_forecast_row(row, 'historical_empirical')
            empirical_row.update({
                'pred_mean': historical_mean,
                'pred_vol': historical_std,
                'history_n': int(len(history)),
            })
            output_rows.append(add_tail_statistics(empirical_row, empirical_tail_values))

            # Rolling Gaussian
            normal_tail_values = {
                alpha: normal_var_and_loss_es(alpha, historical_mean, historical_std)
                for alpha in ALPHAS
            }
            normal_row = base_forecast_row(row, 'rolling_normal')
            normal_row.update({
                'pred_mean': historical_mean,
                'pred_vol': historical_std,
                'history_n': int(len(history)),
                'loc': historical_mean,
                'scale': historical_std,
            })
            output_rows.append(add_tail_statistics(normal_row, normal_tail_values))

            # Rolling Student-t by moments
            t_loc, t_scale, t_nu, t_vol = rolling_t_parameters(history)
            student_tail_values = {
                alpha: student_var_and_loss_es(alpha, t_loc, t_scale, t_nu)
                for alpha in ALPHAS
            }
            student_row = base_forecast_row(row, 'rolling_student_t_mom')
            student_row.update({
                'pred_mean': t_loc,
                'pred_vol': t_vol,
                'history_n': int(len(history)),
                'loc': t_loc,
                'scale': t_scale,
                'nu': t_nu,
            })
            output_rows.append(add_tail_statistics(student_row, student_tail_values))

    return pd.DataFrame(output_rows) if output_rows else empty_forecast_frame()

if RUN_HISTORICAL_BASELINES:
    rolling_forecasts = run_rolling_distribution_baselines(panel_df)
else:
    rolling_forecasts = empty_forecast_frame()

print('Rolling forecast rows:', len(rolling_forecasts))
display(rolling_forecasts.groupby('model_id').size().rename('rows').to_frame())


Rolling baselines:   0%|          | 0/198 [00:00<?, ?it/s]

Rolling forecast rows: 3675


,rows
model_id,
historical_empirical,1225
rolling_normal,1225
rolling_student_t_mom,1225


## 7. Conditional One-Component Normal and Student-$t$ Networks

These are deliberately simpler same-information-set MLP benchmarks. They use all designated firm and macro predictors but impose a single conditional distribution instead of a mixture. For an exact architecture-matched nested comparison, run `3MDN_model.ipynb` with `best_k=1`. Each refit here uses only rows whose target return was already observable before the first forecast month of the block.


In [7]:
class ConditionalDistributionNet(nn.Module):
    def __init__(
        self, input_dim: int, distribution: str, hidden_dims=HIDDEN_DIMS, dropout=DROPOUT
    ):
        super().__init__()
        if distribution not in {'normal', 'student_t'}:
            raise ValueError('distribution must be normal or student_t')
        self.distribution = distribution

        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ELU(),
                nn.Dropout(dropout),
            ])
            previous_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.mu_head = nn.Linear(previous_dim, 1)
        self.sigma_head = nn.Linear(previous_dim, 1)
        self.nu_head = nn.Linear(previous_dim, 1) if distribution == 'student_t' else None

        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)
        nn.init.constant_(self.sigma_head.bias, -2.35)
        if self.nu_head is not None:
            nn.init.zeros_(self.nu_head.weight)
            nn.init.zeros_(self.nu_head.bias)

    def forward(self, x):
        hidden = self.backbone(x)
        mu = self.mu_head(hidden).squeeze(-1)
        sigma = F.softplus(self.sigma_head(hidden).squeeze(-1)) + SIGMA_FLOOR
        if self.nu_head is None:
            nu = None
        else:
            raw_nu = self.nu_head(hidden).squeeze(-1)
            nu = NU_MIN + (NU_MAX - NU_MIN) * torch.sigmoid(raw_nu)
        return mu, sigma, nu

def fit_feature_scaler(training_frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    if not STANDARDIZE_FEATURES_WITH_TRAINING_DATA:
        return np.zeros(len(FEATURE_COLS), dtype=np.float32), np.ones(len(FEATURE_COLS), dtype=np.float32)
    micro_mean = training_frame[MICRO_FEATURES].mean()
    micro_std = training_frame[MICRO_FEATURES].std(ddof=0)
    # Macro variables repeat for every stock. Fit their scaler on unique months,
    # so months with more listed firms do not receive more weight.
    macro_by_month = training_frame[[DATE_COL] + MACRO_FEATURES].drop_duplicates(DATE_COL)
    macro_mean = macro_by_month[MACRO_FEATURES].mean()
    macro_std = macro_by_month[MACRO_FEATURES].std(ddof=0)
    means = pd.concat([micro_mean, macro_mean]).loc[FEATURE_COLS].to_numpy(dtype=np.float32)
    stds = pd.concat([micro_std, macro_std]).loc[FEATURE_COLS].to_numpy(dtype=np.float32)
    stds = np.where(stds < 1e-6, 1.0, stds)
    return means, stds

def transform_features(frame: pd.DataFrame, means: np.ndarray, stds: np.ndarray) -> np.ndarray:
    values = frame[FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
    return (values - means) / stds

def add_month_equal_weights(frame: pd.DataFrame) -> np.ndarray:
    counts = frame.groupby(DATE_COL)[ID_COL].transform('size').to_numpy(dtype=np.float32)
    weights = 1.0 / np.maximum(counts, 1.0)
    return weights / weights.mean()

def split_training_and_validation(history: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    dates = np.sort(history[DATE_COL].unique())
    validation_count = min(VALIDATION_MONTHS, max(1, len(dates) // 4))
    validation_dates = set(dates[-validation_count:])
    train_frame = history.loc[~history[DATE_COL].isin(validation_dates)].copy()
    validation_frame = history.loc[history[DATE_COL].isin(validation_dates)].copy()
    if train_frame.empty or validation_frame.empty:
        raise ValueError('Not enough dates for an internal train/validation split.')
    if train_frame[DATE_COL].nunique() < MIN_NEURAL_TRAIN_MONTHS:
        raise ValueError(
            f'Need at least {MIN_NEURAL_TRAIN_MONTHS} training months before validation; '
            f'found {train_frame[DATE_COL].nunique()}.'
        )
    return train_frame, validation_frame

def distribution_nll(model, x, y, weights):
    mu, sigma, nu = model(x)
    if model.distribution == 'normal':
        log_prob = torch.distributions.Normal(mu, sigma).log_prob(y)
    else:
        log_prob = torch.distributions.StudentT(nu, loc=mu, scale=sigma).log_prob(y)
    return -(weights * log_prob).sum() / weights.sum().clamp_min(1e-8)

def train_conditional_distribution(
    train_frame: pd.DataFrame, validation_frame: pd.DataFrame, distribution: str, block_seed: int
):
    set_seed(block_seed)
    means, stds = fit_feature_scaler(train_frame)

    x_train = torch.tensor(transform_features(train_frame, means, stds), dtype=torch.float32)
    y_train = torch.tensor(train_frame[TARGET_COL].to_numpy(dtype=np.float32), dtype=torch.float32)
    w_train = torch.tensor(add_month_equal_weights(train_frame), dtype=torch.float32)
    train_loader = DataLoader(
        TensorDataset(x_train, y_train, w_train), batch_size=BATCH_SIZE, shuffle=True
    )

    x_validation = torch.tensor(transform_features(validation_frame, means, stds), dtype=torch.float32, device=DEVICE)
    y_validation = torch.tensor(validation_frame[TARGET_COL].to_numpy(dtype=np.float32), dtype=torch.float32, device=DEVICE)
    w_validation = torch.tensor(add_month_equal_weights(validation_frame), dtype=torch.float32, device=DEVICE)

    model = ConditionalDistributionNet(len(FEATURE_COLS), distribution=distribution).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    best_state = None
    best_validation_loss = np.inf
    patience_counter = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for x_batch, y_batch, w_batch in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            w_batch = w_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = distribution_nll(model, x_batch, y_batch, w_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_loss = float(distribution_nll(
                model, x_validation, y_validation, w_validation
            ).cpu())
        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is None:
        raise RuntimeError('Training produced no finite validation loss. Check targets, features, and scales.')
    model.load_state_dict(best_state)
    return model, means, stds, best_validation_loss


In [8]:
def iter_oos_blocks(panel: pd.DataFrame):
    oos_dates = np.array(sorted(
        panel.loc[panel[DATE_COL].between(OOS_START, OOS_END), DATE_COL].unique()
    ))
    for start_position in range(0, len(oos_dates), REFIT_MONTHS):
        block_dates = oos_dates[start_position:start_position + REFIT_MONTHS]
        if len(block_dates) == 0:
            continue
        block_start = pd.Timestamp(block_dates[0])
        history = panel.loc[panel['target_date'] < block_start].copy()
        test_frame = panel.loc[panel[DATE_COL].isin(block_dates)].copy()
        yield start_position, block_start, history, test_frame

def run_conditional_parametric_oos(panel: pd.DataFrame, distribution: str) -> pd.DataFrame:
    model_id = 'conditional_normal_k1' if distribution == 'normal' else 'conditional_student_t_k1'
    output_rows = []

    for block_number, block_start, history, test_frame in iter_oos_blocks(panel):
        train_frame, validation_frame = split_training_and_validation(history)
        model, means, stds, validation_loss = train_conditional_distribution(
            train_frame, validation_frame, distribution, SEED + block_number
        )

        x_test = torch.tensor(
            transform_features(test_frame, means, stds), dtype=torch.float32, device=DEVICE
        )
        model.eval()
        with torch.no_grad():
            mu_tensor, sigma_tensor, nu_tensor = model(x_test)
        mu_values = mu_tensor.cpu().numpy()
        sigma_values = sigma_tensor.cpu().numpy()
        nu_values = None if nu_tensor is None else nu_tensor.cpu().numpy()

        for position, (_, row) in enumerate(test_frame.iterrows()):
            loc = float(mu_values[position])
            scale = float(sigma_values[position])
            output_row = base_forecast_row(row, model_id)
            if distribution == 'normal':
                pred_vol = scale
                tail_values = {
                    alpha: normal_var_and_loss_es(alpha, loc, scale) for alpha in ALPHAS
                }
                output_row.update({'loc': loc, 'scale': scale})
            else:
                nu = float(nu_values[position])
                pred_vol = scale * math.sqrt(nu / (nu - 2.0))
                tail_values = {
                    alpha: student_var_and_loss_es(alpha, loc, scale, nu) for alpha in ALPHAS
                }
                output_row.update({'loc': loc, 'scale': scale, 'nu': nu})
            output_row.update({
                'pred_mean': loc,
                'pred_vol': float(pred_vol),
                'validation_loss': float(validation_loss),
                'refit_date': block_start,
            })
            output_rows.append(add_tail_statistics(output_row, tail_values))

    return pd.DataFrame(output_rows) if output_rows else empty_forecast_frame()

if RUN_CONDITIONAL_NORMAL_K1:
    conditional_normal_forecasts = run_conditional_parametric_oos(panel_df, 'normal')
else:
    conditional_normal_forecasts = empty_forecast_frame()

if RUN_CONDITIONAL_STUDENT_K1:
    conditional_student_forecasts = run_conditional_parametric_oos(panel_df, 'student_t')
else:
    conditional_student_forecasts = empty_forecast_frame()

display(pd.concat([
    conditional_normal_forecasts, conditional_student_forecasts
]).groupby('model_id').size().rename('rows').to_frame())


,rows
model_id,
conditional_normal_k1,1415
conditional_student_t_k1,1415


## 8. Quantile Benchmarks

The primary quantile benchmark is a joint **linear** quantile model trained with pinball loss. The optional MLP changes only the feature map; both models use the same quantile grid, time split, feature scaler, and walk-forward dates. Predicted quantiles are sorted at inference to remove finite-sample crossings.

For a quantile grid $Q(u)$, lower-tail ES is computed as

$$
LES_\alpha=-\frac{1}{\alpha}\int_0^\alpha Q(u)\,du.
$$


In [9]:
class JointQuantileNet(nn.Module):
    def __init__(self, input_dim: int, quantile_count: int, model_kind: str):
        super().__init__()
        if model_kind == 'linear':
            self.network = nn.Linear(input_dim, quantile_count)
        elif model_kind == 'mlp':
            self.network = nn.Sequential(
                nn.Linear(input_dim, HIDDEN_DIMS[0]),
                nn.LayerNorm(HIDDEN_DIMS[0]),
                nn.ELU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1]),
                nn.LayerNorm(HIDDEN_DIMS[1]),
                nn.ELU(),
                nn.Linear(HIDDEN_DIMS[1], quantile_count),
            )
        else:
            raise ValueError('model_kind must be linear or mlp')

    def forward(self, x):
        return self.network(x)

def weighted_pinball_loss(predicted_quantiles, targets, weights, quantile_tensor):
    errors = targets.unsqueeze(1) - predicted_quantiles
    pinball = torch.maximum(quantile_tensor * errors, (quantile_tensor - 1.0) * errors)
    row_loss = pinball.mean(dim=1)
    return (weights * row_loss).sum() / weights.sum().clamp_min(1e-8)

def train_quantile_model(
    train_frame: pd.DataFrame, validation_frame: pd.DataFrame, model_kind: str, block_seed: int
):
    set_seed(block_seed)
    means, stds = fit_feature_scaler(train_frame)
    quantile_tensor = torch.tensor(QUANTILE_LEVELS, dtype=torch.float32, device=DEVICE).view(1, -1)

    x_train = torch.tensor(transform_features(train_frame, means, stds), dtype=torch.float32)
    y_train = torch.tensor(train_frame[TARGET_COL].to_numpy(dtype=np.float32), dtype=torch.float32)
    w_train = torch.tensor(add_month_equal_weights(train_frame), dtype=torch.float32)
    train_loader = DataLoader(
        TensorDataset(x_train, y_train, w_train), batch_size=BATCH_SIZE, shuffle=True
    )

    x_validation = torch.tensor(transform_features(validation_frame, means, stds), dtype=torch.float32, device=DEVICE)
    y_validation = torch.tensor(validation_frame[TARGET_COL].to_numpy(dtype=np.float32), dtype=torch.float32, device=DEVICE)
    w_validation = torch.tensor(add_month_equal_weights(validation_frame), dtype=torch.float32, device=DEVICE)

    model = JointQuantileNet(len(FEATURE_COLS), len(QUANTILE_LEVELS), model_kind).to(DEVICE)
    output_layer = model.network if model_kind == 'linear' else model.network[-1]
    initial_quantiles = np.quantile(train_frame[TARGET_COL], QUANTILE_LEVELS)
    nn.init.zeros_(output_layer.weight)
    with torch.no_grad():
        output_layer.bias.copy_(torch.tensor(initial_quantiles, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    best_state = None
    best_validation_loss = np.inf
    patience_counter = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for x_batch, y_batch, w_batch in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            w_batch = w_batch.to(DEVICE)
            optimizer.zero_grad()
            predicted = model(x_batch)
            loss = weighted_pinball_loss(predicted, y_batch, w_batch, quantile_tensor)
            crossing_penalty = F.relu(predicted[:, :-1] - predicted[:, 1:]).mean()
            total_loss = loss + 0.10 * crossing_penalty
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_prediction = model(x_validation)
            validation_loss = float(weighted_pinball_loss(
                validation_prediction, y_validation, w_validation, quantile_tensor
            ).cpu())
        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is None:
        raise RuntimeError('Quantile training produced no finite validation loss.')
    model.load_state_dict(best_state)
    return model, means, stds, best_validation_loss

def trapezoid_integral(values: np.ndarray, levels: np.ndarray) -> float:
    # NumPy >=2.0 prefers trapezoid; older Colab images expose trapz only.
    integrate = getattr(np, 'trapezoid', None)
    if integrate is None:
        integrate = np.trapz
    return float(integrate(values, levels))

def quantile_grid_moments(quantile_values: np.ndarray) -> tuple[float, float]:
    quantile_values = np.sort(np.asarray(quantile_values, dtype=float))
    full_levels = np.concatenate(([0.0], QUANTILE_LEVELS, [1.0]))
    full_quantiles = np.concatenate(([quantile_values[0]], quantile_values, [quantile_values[-1]]))
    mean_value = trapezoid_integral(full_quantiles, full_levels)
    second_moment = trapezoid_integral(full_quantiles ** 2, full_levels)
    variance = max(second_moment - mean_value ** 2, 0.0)
    return mean_value, math.sqrt(variance)

def quantile_grid_var_and_loss_es(
    quantile_values: np.ndarray, alpha: float
) -> tuple[float, float]:
    quantile_values = np.sort(np.asarray(quantile_values, dtype=float))
    var_value = float(np.interp(alpha, QUANTILE_LEVELS, quantile_values))
    interior_mask = QUANTILE_LEVELS < alpha
    integration_levels = np.concatenate(([0.0], QUANTILE_LEVELS[interior_mask], [alpha]))
    integration_quantiles = np.concatenate((
        [quantile_values[0]], quantile_values[interior_mask], [var_value]
    ))
    tail_mean = trapezoid_integral(integration_quantiles, integration_levels) / alpha
    return var_value, -tail_mean


In [10]:
def run_quantile_oos(panel: pd.DataFrame, model_kind: str) -> pd.DataFrame:
    model_id = 'linear_quantile' if model_kind == 'linear' else 'quantile_mlp'
    output_rows = []

    for block_number, block_start, history, test_frame in iter_oos_blocks(panel):
        train_frame, validation_frame = split_training_and_validation(history)
        model, means, stds, validation_loss = train_quantile_model(
            train_frame, validation_frame, model_kind, SEED + 10_000 + block_number
        )
        x_test = torch.tensor(
            transform_features(test_frame, means, stds), dtype=torch.float32, device=DEVICE
        )
        model.eval()
        with torch.no_grad():
            predicted_quantiles = model(x_test).cpu().numpy()
        predicted_quantiles = np.sort(predicted_quantiles, axis=1)

        for position, (_, row) in enumerate(test_frame.iterrows()):
            quantile_values = predicted_quantiles[position]
            pred_mean, pred_vol = quantile_grid_moments(quantile_values)
            tail_values = {
                alpha: quantile_grid_var_and_loss_es(quantile_values, alpha)
                for alpha in ALPHAS
            }
            output_row = base_forecast_row(row, model_id)
            output_row.update({
                'pred_mean': pred_mean,
                'pred_vol': pred_vol,
                'quantile_levels': json.dumps(QUANTILE_LEVELS.tolist()),
                'quantile_vec': json.dumps(quantile_values.tolist()),
                'validation_loss': float(validation_loss),
                'refit_date': block_start,
            })
            output_rows.append(add_tail_statistics(output_row, tail_values))

    return pd.DataFrame(output_rows) if output_rows else empty_forecast_frame()

if RUN_LINEAR_QUANTILE:
    linear_quantile_forecasts = run_quantile_oos(panel_df, 'linear')
else:
    linear_quantile_forecasts = empty_forecast_frame()

if RUN_QUANTILE_MLP:
    quantile_mlp_forecasts = run_quantile_oos(panel_df, 'mlp')
else:
    quantile_mlp_forecasts = empty_forecast_frame()

display(pd.concat([
    linear_quantile_forecasts, quantile_mlp_forecasts
]).groupby('model_id').size().rename('rows').to_frame())


,rows
model_id,
linear_quantile,1415


## 9. Combine, Validate, and Save

This section performs structural checks only. NLL, CRPS, PIT, VaR coverage, and joint VaR--ES model selection belong in `5PIT_CRPS_Test.ipynb`.


In [11]:
forecast_frames = [
    rolling_forecasts,
    conditional_normal_forecasts,
    conditional_student_forecasts,
    linear_quantile_forecasts,
    quantile_mlp_forecasts,
]
baseline_forecasts = pd.concat(
    [frame for frame in forecast_frames if not frame.empty], ignore_index=True, sort=False
)
baseline_forecasts['forecast_date'] = pd.to_datetime(baseline_forecasts['forecast_date'])
baseline_forecasts['target_date'] = pd.to_datetime(baseline_forecasts['target_date'])
baseline_forecasts = baseline_forecasts.sort_values(
    ['model_id', 'forecast_date', ID_COL]
).reset_index(drop=True)

required_output_cols = [
    ID_COL, 'forecast_date', 'target_date', 'realized_ret', 'model_id',
    'pred_mean', 'pred_vol', 'var_1', 'es_1',
    'var_5', 'es_5', 'var_10', 'es_10'
]
missing_output_cols = [col for col in required_output_cols if col not in baseline_forecasts.columns]
if missing_output_cols:
    raise KeyError(f'Unified output is missing columns: {missing_output_cols}')

if baseline_forecasts.duplicated(['model_id', ID_COL, 'forecast_date']).any():
    raise ValueError('Duplicate model-stock-forecast_date rows found.')

finite_cols = ['pred_mean', 'pred_vol', 'var_1', 'es_1', 'var_5', 'es_5', 'var_10', 'es_10']
if not np.isfinite(baseline_forecasts[finite_cols].to_numpy(dtype=float)).all():
    raise ValueError('Non-finite distribution statistics found in baseline output.')

tolerance = 1e-8
var_order_ok = (
    (baseline_forecasts['var_1'] <= baseline_forecasts['var_5'] + tolerance)
    & (baseline_forecasts['var_5'] <= baseline_forecasts['var_10'] + tolerance)
).mean()
es_order_ok = (
    (baseline_forecasts['es_1'] + tolerance >= baseline_forecasts['es_5'])
    & (baseline_forecasts['es_5'] + tolerance >= baseline_forecasts['es_10'])
).mean()

summary = baseline_forecasts.groupby('model_id').agg(
    rows=(ID_COL, 'size'),
    first_forecast=('forecast_date', 'min'),
    last_forecast=('forecast_date', 'max'),
    mean_predicted_return=('pred_mean', 'mean'),
    mean_predicted_volatility=('pred_vol', 'mean'),
    mean_es_5=('es_5', 'mean'),
).sort_index()

print(f'VaR ordering pass rate: {var_order_ok:.4%}')
print(f'Loss-ES ordering pass rate: {es_order_ok:.4%}')
display(summary)
display(baseline_forecasts[required_output_cols].head(10))

if var_order_ok < 0.999 or es_order_ok < 0.999:
    raise AssertionError('VaR/ES alpha ordering check failed for too many rows.')

should_save = (not USE_SYNTHETIC_DATA) or SAVE_SYNTHETIC_OUTPUT
if should_save:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    baseline_forecasts.to_parquet(OUTPUT_PATH, index=False)
    config_to_save = {
        'environment': ENV,
        'run_mode': RUN_MODE,
        'micro_data_path': str(MICRO_DATA_PATH),
        'macro_data_path': str(MACRO_DATA_PATH),
        'output_path': str(OUTPUT_PATH),
        'oos_start': str(OOS_START.date()),
        'oos_end': str(pd.Timestamp(OOS_END).date()),
        'micro_features': MICRO_FEATURES,
        'rank_micro_features': RANK_MICRO_FEATURES,
        'raw_scale_micro_features': RAW_SCALE_MICRO_FEATURES,
        'macro_features': MACRO_FEATURES,
        'alphas': list(ALPHAS),
        'quantile_levels': QUANTILE_LEVELS.tolist(),
        'history_window_months': HISTORY_WINDOW_MONTHS,
        'min_history_months': MIN_HISTORY_MONTHS,
        'refit_months': REFIT_MONTHS,
        'validation_months': VALIDATION_MONTHS,
        'seed': SEED,
    }
    CONFIG_PATH.write_text(json.dumps(config_to_save, indent=2), encoding='utf-8')
    print('Saved forecasts to:', OUTPUT_PATH)
    print('Saved config to:', CONFIG_PATH)
else:
    print('Synthetic smoke-test output was not saved.')


VaR ordering pass rate: 100.0000%
Loss-ES ordering pass rate: 100.0000%


,rows,first_forecast,last_forecast,mean_predicted_return,mean_predicted_volatility,mean_es_5
model_id,,,,,,
conditional_normal_k1,1415,2007-01-31,2008-12-31,0.014839,0.120538,0.233796
conditional_student_t_k1,1415,2007-01-31,2008-12-31,0.007661,0.118077,0.242723
historical_empirical,1225,2007-01-31,2008-12-31,0.018287,0.123674,0.221489
linear_quantile,1415,2007-01-31,2008-12-31,0.020065,0.151722,0.279038
rolling_normal,1225,2007-01-31,2008-12-31,0.018287,0.123674,0.236817
rolling_student_t_mom,1225,2007-01-31,2008-12-31,0.018287,0.123674,0.251499


,permno,forecast_date,target_date,realized_ret,model_id,pred_mean,pred_vol,var_1,es_1,var_5,es_5,var_10,es_10
0,11267,2007-01-31,2007-02-28,-0.034116,conditional_normal_k1,0.013366,0.109932,-0.242374,0.279626,-0.167456,0.213392,-0.127518,0.179563
1,11353,2007-01-31,2007-02-28,-0.069106,conditional_normal_k1,0.003431,0.089539,-0.204868,0.235210,-0.143848,0.181263,-0.111318,0.153709
2,11513,2007-01-31,2007-02-28,-0.014173,conditional_normal_k1,0.018572,0.067756,-0.139053,0.162013,-0.092877,0.121190,-0.068261,0.100339
3,13688,2007-01-31,2007-02-28,-0.005356,conditional_normal_k1,0.003994,0.069289,-0.157197,0.180677,-0.109977,0.138930,-0.084804,0.117608
4,16126,2007-01-31,2007-02-28,0.011741,conditional_normal_k1,0.020717,0.141404,-0.308238,0.356155,-0.211872,0.270959,-0.160499,0.227444
5,17137,2007-01-31,2007-02-28,0.008783,conditional_normal_k1,-0.002851,0.084960,-0.200497,0.229287,-0.142597,0.178099,-0.111731,0.151954
6,20336,2007-01-31,2007-02-28,0.076023,conditional_normal_k1,0.015146,0.143102,-0.317758,0.366250,-0.220235,0.280031,-0.168246,0.235995
7,22374,2007-01-31,2007-02-28,-0.045858,conditional_normal_k1,0.018525,0.130585,-0.285261,0.329512,-0.196268,0.250834,-0.148826,0.210649
8,23114,2007-01-31,2007-02-28,0.027766,conditional_normal_k1,-0.000262,0.067283,-0.156786,0.179586,-0.110933,0.139048,-0.086489,0.118343
9,27713,2007-01-31,2007-02-28,-0.111282,conditional_normal_k1,0.012088,0.093556,-0.205557,0.237260,-0.141798,0.180892,-0.107809,0.152102


Saved forecasts to: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output/baseline/baseline_oos_forecasts_quick.parquet
Saved config to: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output/baseline/baseline_config_quick.json


In [13]:
baseline_forecasts.sample(10)

,permno,forecast_date,target_date,realized_ret,model_id,pred_mean,pred_vol,history_n,var_1,es_1,var_5,es_5,var_10,es_10,loc,scale,nu,validation_loss,refit_date,quantile_levels,quantile_vec
6389,81298,2008-06-30,2008-07-31,-0.088235,rolling_normal,0.001254,0.059442,120.0,-0.137030,0.157173,-0.096520,0.121359,-0.074924,0.103066,0.001254,0.059442,NaN,NaN,NaT,NaN,NaN
3371,13688,2007-11-30,2007-12-31,-0.061006,historical_empirical,0.010509,0.094472,120.0,-0.257323,0.283041,-0.156477,0.227560,-0.073463,0.173742,NaN,NaN,NaN,NaN,NaT,NaN,NaN
5211,91688,2008-07-31,2008-08-31,-0.021315,linear_quantile,0.033078,0.118004,NaN,-0.264109,0.328392,-0.140534,0.216207,-0.075782,0.162183,NaN,NaN,NaN,0.011734,2008-01-31,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.3928845226764679, -0.35620003938674927, -0..."
2608,85571,2008-08-31,2008-09-30,-0.031296,conditional_student_t_k1,0.004221,0.291944,NaN,-0.702172,0.835107,-0.472403,0.615057,-0.360496,0.513054,0.004221,0.272456,15.498255,-0.884019,2008-01-31,NaN,NaN
7476,86786,2008-03-31,2008-04-30,0.082305,rolling_student_t_mom,0.028163,0.147740,87.0,-0.357629,0.484515,-0.201852,0.303024,-0.140043,0.235191,0.028163,0.113590,4.891518,NaN,NaT,NaN,NaN
7316,85047,2007-12-31,2008-01-31,0.045775,rolling_student_t_mom,0.006068,0.063917,120.0,-0.154494,0.193379,-0.096788,0.133229,-0.071130,0.107919,0.006068,0.055168,7.841785,NaN,NaT,NaN,NaN
4968,87269,2008-03-31,2008-04-30,-0.136068,linear_quantile,0.306602,0.232198,NaN,-0.103086,0.555807,0.086416,0.080206,0.139099,-0.016276,NaN,NaN,NaN,0.011734,2008-01-31,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.363386631011963, -1.1577060222625732, -0.1..."
1818,61807,2007-07-31,2007-08-31,-0.005838,conditional_student_t_k1,0.003966,0.137853,NaN,-0.329427,0.391975,-0.221119,0.288351,-0.168310,0.240261,0.003966,0.128765,15.686440,-1.079500,2007-01-31,NaN,NaN
3081,83534,2007-05-31,2007-06-30,-0.033563,historical_empirical,0.030862,0.212674,116.0,-0.448615,0.629598,-0.193359,0.385360,-0.142985,0.280994,NaN,NaN,NaN,NaN,NaT,NaN,NaN
376,90086,2007-06-30,2007-07-31,0.019565,conditional_normal_k1,0.019670,0.066865,NaN,-0.135881,0.158540,-0.090313,0.118253,-0.066021,0.097677,0.019670,0.066865,NaN,-1.046290,2007-01-31,NaN,NaN


## 10. Handoff to Forecast Evaluation

Before the real run:

1. choose `RUN_MODE='quick'` for a bounded test or `RUN_MODE='full'` for production;
2. on Kaggle, attach the dataset containing both parquet files; on Colab, place them under `COLAB_DATA_DIR`;
3. confirm the explicit micro/macro feature lists and final OOS dates;
4. run top-to-bottom and keep the generated config JSON with the Parquet output.

The next notebook should compare these forecasts with MDN forecasts using monthly aggregated NLL/CRPS/tail-CRPS, PIT diagnostics, VaR coverage, and a joint VaR--ES score. Do not select a benchmark or MDN using portfolio returns.
